# Dog Pose Estimation

Uses DeepLabCut for dog pose estimation.

In [1]:
import numpy as np
import pandas as pd
import deeplabcut
import tensorflow as tf
from dlclibrary import download_huggingface_model

/supernova/data/home/lillian/chihuahua/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1781274138.109958  438892 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781274138.147408  438892 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781274138.821671  438892 port.cc:153] oneDNN custom operations are on. You may see slightly different nume

In [2]:
download_huggingface_model("full_dog", "dog_model")

Loading.... full_dog


In [3]:
results = deeplabcut.video_inference_superanimal(
    ["files/dogs1.mp4"],
    "superanimal_quadruped",
    model_name="hrnet_w32",
    detector_name="fasterrcnn_resnet50_fpn_v2"
)

Running video inference on ['files/dogs1.mp4'] with superanimal_quadruped_hrnet_w32
Using pytorch for model hrnet_w32
Processing video files/dogs1.mp4
Starting to analyze files/dogs1.mp4
Video metadata: 
  Overall # of frames:    150
  Duration of video [s]:  5.00
  fps:                    30.0
  resolution:             w=1280, h=720

Running detector with batch size 1


100%|██████████| 150/150 [00:04<00:00, 31.98it/s]


Running pose prediction with batch size 1


100%|██████████| 150/150 [00:09<00:00, 15.50it/s]


Saving results to files
Saving results in files/dogs1_superanimal_quadruped_hrnet_w32_fasterrcnn_resnet50_fpn_v2.h5 and files/dogs1_superanimal_quadruped_hrnet_w32_fasterrcnn_resnet50_fpn_v2_full.pickle


/supernova/data/home/lillian/chihuahua/lib/python3.12/site-packages/deeplabcut/utils/make_labeled_video.py:138: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  nbpts_per_ind = Dataframe.groupby(level="individuals", axis=1).size().values // 3


Duration of video [s]: 5.0, recorded with 30.0 fps!
Overall # of frames: 150 with cropped frame dimensions: 1280 720
Generating frames and creating video.


100%|██████████| 150/150 [00:01<00:00, 80.06it/s]


Video with predictions was saved as files/dogs1_superanimal_quadruped_hrnet_w32_fasterrcnn_resnet50_fpn_v2_labeled_before_adapt.mp4


In [ ]:
def mouth_openness(slice):
    individuals = slice.columns.get_level_values("individuals").unique()

    results = {}

    for ind in individuals:
        ind_df = slice.xs(ind, level="individuals", axis=1)
        ind_df = ind_df.droplevel(0, axis=1)

        ux = ind_df[("upper_jaw", "x")]
        uy = ind_df[("upper_jaw", "y")]
        lx = ind_df[("lower_jaw", "x")]
        ly = ind_df[("lower_jaw", "y")]

        openness = np.sqrt((ux - lx) ** 2 + (uy - ly) ** 2)

        xs = ind_df.xs("x", level="coords", axis=1)
        ys = ind_df.xs("y", level="coords", axis=1)

        left = xs.min().min()
        right = xs.max().max()
        top = ys.min().min()
        bottom = ys.max().max()

        results[ind] = {
            "openness_mean": openness.mean(),
            "left": left,
            "right": right,
            "top": top,
            "bottom": bottom,
        }

    return results

def cut_dataframe(df, fps, timestamp):
    frame_id = int(timestamp * fps)

    start = max(0, frame_id - 10)
    end = frame_id + 10

    slice = df.iloc[start:end]
    return slice


slice = cut_dataframe(results['files/dogs1.mp4'], 30, 2.5)

print(slice)

results = mouth_openness(slice)

print(results)

scorer      superanimal_quadruped_hrnet_w32_fasterrcnn_resnet50_fpn_v2  \
individuals                                                    animal0   
bodyparts                                                         nose   
coords                                                               x   
65                                                  847.179688           
66                                                  709.125000           
67                                                 1126.367188           
68                                                 1123.710938           
69                                                 1123.070312           
70                                                 1119.046875           
71                                                 1118.046875           
72                                                 1114.250000           
73                                                 1110.070312           
74                                    

In [6]:
print(slice.columns.get_level_values("bodyparts").unique())

Index(['nose', 'upper_jaw', 'lower_jaw', 'mouth_end_right', 'mouth_end_left',
       'right_eye', 'right_earbase', 'right_earend', 'right_antler_base',
       'right_antler_end', 'left_eye', 'left_earbase', 'left_earend',
       'left_antler_base', 'left_antler_end', 'neck_base', 'neck_end',
       'throat_base', 'throat_end', 'back_base', 'back_end', 'back_middle',
       'tail_base', 'tail_end', 'front_left_thai', 'front_left_knee',
       'front_left_paw', 'front_right_thai', 'front_right_knee',
       'front_right_paw', 'back_left_paw', 'back_left_thai', 'back_right_thai',
       'back_left_knee', 'back_right_knee', 'back_right_paw', 'belly_bottom',
       'body_middle_right', 'body_middle_left'],
      dtype='object', name='bodyparts')
